# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mfaiqdev/MLinternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from huggingface_hub import hf_hub_download
import pandas as pd

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset"
)

df = pd.read_parquet(file_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)

df.head()

fact_content_daily_performance/month=202(…):   0%|          | 0.00/124M [00:00<?, ?B/s]

Dataset loaded successfully
Shape: (9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My unit of analysis is **one content item for one client on one reporting date**.

Each row represents a daily performance snapshot of a single content item, identified by `content_hash_id`, belonging to a client identified by `client_hash_id`.

The table used for this lane is:

`fact_content_daily_performance`

This table contains daily Search Console and Analytics performance signals.

For development, I use the middle-panel month **2026-03** instead of the final month. This prevents using future information and follows the rule that the final month should be treated as a sealed test period.

The time window contains observed historical performance signals available at the reporting date.

In [17]:
grain_check = (
    df.groupby(
        ["report_date", "client_hash_id", "content_hash_id"]
    )
    .size()
    .reset_index(name="rows")
)

duplicates = grain_check[grain_check["rows"] > 1]

print("Duplicate grain rows:")
display(duplicates.head())

Duplicate grain rows:


,report_date,client_hash_id,content_hash_id,rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features

The features selected for the Content Opportunity Scoring lane are:

- **gsc_impressions** — shows search visibility of a content item before a refresh decision.
- **gsc_clicks** — shows organic search traffic generated by the content.
- **gsc_avg_position** — represents search ranking visibility.
- **ga4_sessions** — measures user visits after content discovery.
- **ga4_engaged_sessions** — measures meaningful user interactions.

These features are available at the decision moment because they describe historical performance signals that exist before deciding which content should be reviewed.

### Label / Proxy

The target of this lane is to rank content items that may require refresh attention.

The proxy label will represent future content performance change measured from observed performance trends.

The label is derived from outcome behaviour after the prediction moment and is not included as a feature.

This prevents the model from learning the answer directly.

The label represents an observed outcome and is not used as a feature.

### Context

The following fields are used only for identification, filtering, and grouping:

- **content_hash_id**
- **client_hash_id**
- **report_date**

These fields define the data grain but are not meaningful content characteristics.

### Excluded

The following information is deliberately excluded:

- Future performance measurements
- Any label-derived columns
- Final test-period information

These fields are excluded because they would introduce data leakage and create unrealistic model performance.

In [12]:
selected_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

feature_frame = df[selected_fields]

feature_frame.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,NaN,NaN


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The following checks verify that the data contract matches the warehouse data.

The queries confirm:

1. The grain is one row per content item, client, and reporting date.
2. The selected month contains the expected number of rows and reporting dates.
3. Required data availability signals are present.

In [13]:
grain_check = (
    df.groupby(
        ["report_date", "client_hash_id", "content_hash_id"]
    )
    .size()
    .reset_index(name="rows")
)

print("Number of duplicate grain rows:",
      len(grain_check[grain_check["rows"] > 1]))

Number of duplicate grain rows: 0


In [14]:
print("Total rows:", len(df))
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

Total rows: 9841378
Start date: 2026-03-01
End date: 2026-03-31


In [15]:
gsc_available = df[df["gsc_data_available"].eq(True)]

print("Rows where GSC data is available:",
      len(gsc_available))

Rows where GSC data is available: 3611061


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset supports observed analysis and decision-support recommendations, but it cannot prove causal relationships.

The data cannot tell whether a content update directly caused an improvement in rankings or traffic because search performance is influenced by many external factors.

Client history depth is not perfectly balanced, meaning some clients have more historical observations than others.

Some Search Console and Analytics values may be unavailable before tracking was enabled. Missing values therefore do not always mean zero activity.

Additionally, this notebook only analyzes the March 2026 slice of the warehouse. A model trained on this slice may not represent seasonal changes or future periods.

The ranking output should be treated as directional guidance for content review rather than a guarantee of future search performance.

In [16]:
# Check data availability limitations

print("Rows with missing GSC availability flag:")
print(df["gsc_data_available"].isna().sum())

print("\nRows with missing GA4 availability flag:")
print(df["ga4_data_available"].isna().sum())

print("\nGSC available rows:")
print(df["gsc_data_available"].eq(True).sum())

print("\nGA4 available rows:")
print(df["ga4_data_available"].eq(True).sum())

Rows with missing GSC availability flag:
0

Rows with missing GA4 availability flag:
3018741

GSC available rows:
3611061

GA4 available rows:
413966


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.